# NLP Evaluation Metrics

**Interview Focus**: BLEU limitations, ROUGE variants, perplexity interpretation, BERTScore concept.

**Key Concepts**:
- BLEU: n-gram precision with brevity penalty (machine translation)
- ROUGE: recall-oriented n-gram overlap (summarization)
- Perplexity: how surprised is the model? (language modeling)
- BERTScore: semantic similarity via contextual embeddings

**Math Notes**:
- BLEU = BP * exp(sum of weighted log-precisions)
- ROUGE-L uses Longest Common Subsequence
- Perplexity = exp(cross-entropy) = $2^{H}$ where H = avg negative log-likelihood

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

## 1. BLEU Score from Scratch

BLEU (Bilingual Evaluation Understudy) measures how many n-grams in the candidate translation appear in the reference.

**Modified precision** for n-gram order n:
$$p_n = \frac{\sum_{\text{n-gram} \in C} \min(\text{count}_{C}(\text{n-gram}), \max_{R}\text{count}_{R}(\text{n-gram}))}{\sum_{\text{n-gram} \in C} \text{count}_{C}(\text{n-gram})}$$

**Brevity Penalty**: penalizes short translations.
$$BP = \begin{cases} 1 & \text{if } c > r \\ e^{1 - r/c} & \text{if } c \leq r \end{cases}$$

**BLEU** = BP * exp($\sum_{n=1}^{N} w_n \log p_n$) with uniform weights $w_n = 1/N$.

In [ ]:
def get_ngrams(tokens, n):
    """Extract n-grams from a token list."""
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]


def modified_precision(candidate, references, n):
    """Compute modified n-gram precision (clipped counts).
    
    For each n-gram in candidate, count is clipped to max count in any reference.
    This prevents gaming by repeating common n-grams.
    """
    candidate_ngrams = get_ngrams(candidate, n)
    if len(candidate_ngrams) == 0:
        return 0.0
    
    candidate_counts = Counter(candidate_ngrams)
    
    # For each n-gram, find max count across all references
    max_ref_counts = Counter()
    for ref in references:
        ref_counts = Counter(get_ngrams(ref, n))
        for ngram, count in ref_counts.items():
            max_ref_counts[ngram] = max(max_ref_counts[ngram], count)
    
    # Clip candidate counts
    clipped_total = 0
    total = 0
    for ngram, count in candidate_counts.items():
        clipped_total += min(count, max_ref_counts.get(ngram, 0))
        total += count
    
    return clipped_total / total


def brevity_penalty(candidate, references):
    """Brevity penalty: penalize if candidate is shorter than closest reference."""
    c = len(candidate)
    # Choose reference length closest to candidate length
    ref_lengths = [len(ref) for ref in references]
    r = min(ref_lengths, key=lambda x: (abs(x - c), x))
    
    if c > r:
        return 1.0
    elif c == 0:
        return 0.0
    else:
        return np.exp(1 - r / c)


def bleu_score(candidate, references, max_n=4, weights=None):
    """Compute BLEU score.
    
    Args:
        candidate: list of tokens
        references: list of list of tokens
        max_n: maximum n-gram order
        weights: weights for each n-gram order (default: uniform)
    """
    if weights is None:
        weights = [1.0 / max_n] * max_n
    
    bp = brevity_penalty(candidate, references)
    
    # Compute weighted log precision
    log_precisions = []
    precisions = []
    for n in range(1, max_n + 1):
        p = modified_precision(candidate, references, n)
        precisions.append(p)
        if p > 0:
            log_precisions.append(weights[n-1] * np.log(p))
        else:
            # If any precision is 0, BLEU is 0
            return 0.0, bp, precisions
    
    score = bp * np.exp(sum(log_precisions))
    return score, bp, precisions


# Demo
reference1 = "the cat is on the mat".split()
reference2 = "there is a cat on the mat".split()
references = [reference1, reference2]

candidate_good = "the cat is on the mat".split()
candidate_ok   = "the the the the the the".split()
candidate_bad  = "a dog sat on a log".split()
candidate_short = "the cat".split()

print("Reference 1:", ' '.join(reference1))
print("Reference 2:", ' '.join(reference2))
print()

for name, cand in [('Perfect', candidate_good), ('Repetitive', candidate_ok),
                    ('Wrong', candidate_bad), ('Too short', candidate_short)]:
    score, bp, precs = bleu_score(cand, references)
    prec_str = ', '.join(f'{p:.3f}' for p in precs)
    print(f"{name:>12}: '{' '.join(cand)}'")
    print(f"             BLEU={score:.4f}, BP={bp:.4f}, P1..P4=[{prec_str}]")
    print()

In [ ]:
# Why modified precision? Demonstrate the "the the the" problem
candidate_gaming = "the the the the the the".split()
ref = ["the cat is on the mat".split()]

# Unmodified precision: count all 'the' occurrences
unigrams_cand = get_ngrams(candidate_gaming, 1)
unigrams_ref = get_ngrams(ref[0], 1)
naive_matches = sum(1 for ug in unigrams_cand if ug in unigrams_ref)
naive_precision = naive_matches / len(unigrams_cand)

# Modified precision: clip to max reference count
mod_precision = modified_precision(candidate_gaming, ref, 1)

print(f"Candidate: '{' '.join(candidate_gaming)}'")
print(f"Reference: '{' '.join(ref[0])}'")
print(f"\nNaive unigram precision:    {naive_precision:.3f} (6/6 — 'the' matches every time!)")
print(f"Modified unigram precision: {mod_precision:.3f} (2/6 — 'the' appears max 2 times in ref)")
print("\nModified precision prevents gaming by clipping counts.")

## 2. BLEU: Brevity Penalty Deep Dive

In [ ]:
# Visualize brevity penalty
ref_length = 20
candidate_lengths = np.arange(1, 40)
bps = [np.exp(1 - ref_length / c) if c <= ref_length else 1.0 for c in candidate_lengths]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(candidate_lengths, bps, 'b-', linewidth=2)
ax.axvline(x=ref_length, color='red', linestyle='--', alpha=0.7, label=f'Reference length = {ref_length}')
ax.axhline(y=1.0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Candidate Length')
ax.set_ylabel('Brevity Penalty')
ax.set_title('Brevity Penalty vs Candidate Length')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

print("BP = 1 if candidate >= reference length (no penalty for being too long).")
print("BP decays exponentially as candidate gets shorter than reference.")
print("Without BP, a system could achieve high precision by outputting very short, safe translations.")

## 3. ROUGE from Scratch

ROUGE (Recall-Oriented Understudy for Gisting Evaluation) is recall-focused (vs BLEU's precision focus).

- **ROUGE-N**: n-gram recall between candidate and reference
- **ROUGE-L**: longest common subsequence (captures sentence-level structure)

In [ ]:
def rouge_n(candidate, reference, n=1):
    """ROUGE-N: n-gram recall, precision, and F1.
    
    Recall = matched n-grams / reference n-grams
    Precision = matched n-grams / candidate n-grams
    """
    cand_ngrams = Counter(get_ngrams(candidate, n))
    ref_ngrams = Counter(get_ngrams(reference, n))
    
    if len(ref_ngrams) == 0 or len(cand_ngrams) == 0:
        return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
    
    # Count overlapping n-grams (clipped)
    overlap = 0
    for ngram, count in ref_ngrams.items():
        overlap += min(count, cand_ngrams.get(ngram, 0))
    
    recall = overlap / sum(ref_ngrams.values())
    precision = overlap / sum(cand_ngrams.values())
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {'precision': precision, 'recall': recall, 'f1': f1}


def lcs_length(x, y):
    """Compute length of Longest Common Subsequence using DP."""
    m, n = len(x), len(y)
    dp = np.zeros((m + 1, n + 1), dtype=int)
    
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if x[i-1] == y[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    
    return dp[m][n]


def rouge_l(candidate, reference):
    """ROUGE-L: LCS-based F1.
    
    Recall = LCS(cand, ref) / len(ref)
    Precision = LCS(cand, ref) / len(cand)
    """
    if len(candidate) == 0 or len(reference) == 0:
        return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
    
    lcs = lcs_length(candidate, reference)
    recall = lcs / len(reference)
    precision = lcs / len(candidate)
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {'precision': precision, 'recall': recall, 'f1': f1}


# Demo: summarization evaluation
reference_summary = "the cat sat on the mat in the living room".split()
candidate_summaries = {
    'Good':     "the cat sat on the mat".split(),
    'Reworded': "a cat was sitting on a mat".split(),
    'Partial':  "the cat is happy".split(),
    'Wrong':    "dogs run in the park".split(),
}

print(f"Reference: '{' '.join(reference_summary)}'\n")
print(f"{'Candidate':<12} {'ROUGE-1 F1':>12} {'ROUGE-2 F1':>12} {'ROUGE-L F1':>12}")
print("-" * 50)
for name, cand in candidate_summaries.items():
    r1 = rouge_n(cand, reference_summary, n=1)
    r2 = rouge_n(cand, reference_summary, n=2)
    rl = rouge_l(cand, reference_summary)
    print(f"{name:<12} {r1['f1']:>12.4f} {r2['f1']:>12.4f} {rl['f1']:>12.4f}")

In [ ]:
# Detailed breakdown for one example
cand = "the cat sat on the mat".split()
ref = "the cat sat on the mat in the living room".split()

print(f"Candidate: '{' '.join(cand)}'")
print(f"Reference: '{' '.join(ref)}'")
print()

for n, name in [(1, 'ROUGE-1'), (2, 'ROUGE-2')]:
    result = rouge_n(cand, ref, n)
    cand_ng = Counter(get_ngrams(cand, n))
    ref_ng = Counter(get_ngrams(ref, n))
    overlap = sum(min(count, cand_ng.get(ng, 0)) for ng, count in ref_ng.items())
    print(f"{name}:")
    print(f"  Candidate {n}-grams: {sum(cand_ng.values())}, Reference {n}-grams: {sum(ref_ng.values())}")
    print(f"  Overlap: {overlap}")
    print(f"  Recall:    {result['recall']:.4f} ({overlap}/{sum(ref_ng.values())})")
    print(f"  Precision: {result['precision']:.4f} ({overlap}/{sum(cand_ng.values())})")
    print(f"  F1:        {result['f1']:.4f}")
    print()

rl = rouge_l(cand, ref)
lcs = lcs_length(cand, ref)
print(f"ROUGE-L:")
print(f"  LCS length: {lcs}")
print(f"  Recall:    {rl['recall']:.4f} ({lcs}/{len(ref)})")
print(f"  Precision: {rl['precision']:.4f} ({lcs}/{len(cand)})")
print(f"  F1:        {rl['f1']:.4f}")

## 4. ROUGE-L vs ROUGE-N: Key Difference

In [ ]:
# ROUGE-L captures non-contiguous matches that ROUGE-N misses
ref = "police killed the gunman".split()

cand1 = "police killed the gunman".split()      # exact match
cand2 = "police the gunman killed".split()       # reordered
cand3 = "the gunman was killed by police".split()  # passive voice

print(f"Reference: '{' '.join(ref)}'\n")

for name, cand in [('Exact', cand1), ('Reordered', cand2), ('Passive', cand3)]:
    r1 = rouge_n(cand, ref, 1)
    r2 = rouge_n(cand, ref, 2)
    rl = rouge_l(cand, ref)
    lcs = lcs_length(cand, ref)
    print(f"{name:>10}: '{' '.join(cand)}'")
    print(f"           ROUGE-1 F1={r1['f1']:.3f}, ROUGE-2 F1={r2['f1']:.3f}, "
          f"ROUGE-L F1={rl['f1']:.3f} (LCS={lcs})")

print("\nROUGE-L finds the longest common subsequence, capturing word overlap")
print("even when word order differs. ROUGE-2 is strict about bigram order.")

## 5. Perplexity

Perplexity measures how well a language model predicts a test set.

$$\text{Perplexity} = \exp\left(-\frac{1}{N} \sum_{i=1}^{N} \log p(w_i | w_{<i})\right) = \exp(\text{cross-entropy})$$

Intuition: "the model is as confused as if it had to choose uniformly among PPL options at each step."
- PPL = 1: perfect prediction
- PPL = V: random guessing (V = vocab size)

In [ ]:
def perplexity_from_logprobs(log_probs):
    """Compute perplexity from per-token log probabilities.
    
    log_probs: array of log p(w_i | w_{<i}) for each token
    """
    avg_neg_logprob = -np.mean(log_probs)
    return np.exp(avg_neg_logprob)


def perplexity_from_cross_entropy(cross_entropy_loss):
    """Perplexity = exp(cross-entropy loss).
    
    This is how it is computed in practice with neural LMs:
    the cross-entropy loss from training IS the average negative log-likelihood.
    """
    return np.exp(cross_entropy_loss)


# Simulate log-probabilities from different quality language models
np.random.seed(42)
n_tokens = 100
vocab_size = 10000

# Good LM: assigns high probability to correct tokens
logprobs_good = np.log(np.random.beta(8, 2, n_tokens))  # mostly high probs

# Mediocre LM
logprobs_med = np.log(np.random.beta(2, 3, n_tokens))

# Bad LM: near-random
logprobs_bad = np.log(np.random.uniform(1e-4, 1.0/100, n_tokens))

# Uniform random baseline
logprobs_random = np.full(n_tokens, np.log(1.0 / vocab_size))

print(f"{'Model':<15} {'Avg log P':>12} {'Cross-Entropy':>15} {'Perplexity':>12}")
print("-" * 57)
for name, lp in [('Good LM', logprobs_good), ('Mediocre LM', logprobs_med),
                  ('Bad LM', logprobs_bad), ('Random', logprobs_random)]:
    ce = -np.mean(lp)
    ppl = perplexity_from_logprobs(lp)
    print(f"{name:<15} {np.mean(lp):>12.4f} {ce:>15.4f} {ppl:>12.1f}")

print(f"\nRandom baseline perplexity = vocab size = {vocab_size}")
print("Lower perplexity = better language model.")

In [ ]:
# Visualize: cross-entropy loss to perplexity mapping
ce_values = np.linspace(0.1, 10, 100)
ppl_values = np.exp(ce_values)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(ce_values, ppl_values, 'b-', linewidth=2)
ax.set_xlabel('Cross-Entropy Loss')
ax.set_ylabel('Perplexity')
ax.set_title('Perplexity = exp(Cross-Entropy)')
ax.grid(True, alpha=0.3)
# Mark common values
for ce, label in [(2.0, 'CE=2, PPL=7.4'), (4.0, 'CE=4, PPL=54.6'), (6.0, 'CE=6, PPL=403')]:
    ax.annotate(label, xy=(ce, np.exp(ce)), fontsize=8,
                arrowprops=dict(arrowstyle='->', color='red'), 
                xytext=(ce+0.5, np.exp(ce)+1000))

# Per-token probability distribution for different models
ax = axes[1]
ax.hist(np.exp(logprobs_good), bins=30, alpha=0.6, label='Good LM', color='#2196F3', density=True)
ax.hist(np.exp(logprobs_med), bins=30, alpha=0.6, label='Mediocre LM', color='#FF9800', density=True)
ax.hist(np.exp(logprobs_bad), bins=30, alpha=0.6, label='Bad LM', color='#F44336', density=True)
ax.set_xlabel('Per-Token Probability')
ax.set_ylabel('Density')
ax.set_title('Distribution of Token Probabilities')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. BERTScore: Concept

BERTScore computes semantic similarity between candidate and reference using contextual embeddings (BERT).

**How it works**:
1. Get BERT embeddings for each token in candidate and reference
2. Compute cosine similarity between all token pairs
3. Greedy matching: each token matches to its most similar counterpart
4. Average matched similarities to get precision/recall/F1

**Advantages over BLEU/ROUGE**:
- Captures semantic similarity ("big" ~ "large")
- Robust to paraphrasing
- Contextual: same word gets different embeddings in different contexts

We illustrate the core idea with simple embeddings (not actual BERT).

In [ ]:
def cosine_similarity(a, b):
    """Cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)


def bertscore_simplified(cand_embeddings, ref_embeddings):
    """Simplified BERTScore using pre-computed embeddings.
    
    In practice, you would use a BERT model to get contextual embeddings.
    Here we demonstrate the matching algorithm.
    """
    n_cand = len(cand_embeddings)
    n_ref = len(ref_embeddings)
    
    # Compute all pairwise cosine similarities
    sim_matrix = np.zeros((n_cand, n_ref))
    for i in range(n_cand):
        for j in range(n_ref):
            sim_matrix[i, j] = cosine_similarity(cand_embeddings[i], ref_embeddings[j])
    
    # Precision: for each candidate token, find best match in reference
    precision = np.mean(np.max(sim_matrix, axis=1))
    
    # Recall: for each reference token, find best match in candidate
    recall = np.mean(np.max(sim_matrix, axis=0))
    
    # F1
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {'precision': precision, 'recall': recall, 'f1': f1, 'sim_matrix': sim_matrix}


# Simulate word embeddings (normally from BERT)
# Make similar words have similar embeddings
np.random.seed(42)
dim = 50

# Create a simple word embedding table
word_embeddings = {}
base_vectors = {
    'cat': np.random.randn(dim),
    'dog': None,  # will be similar to cat
    'sat': np.random.randn(dim),
    'sitting': None,  # similar to sat
    'on': np.random.randn(dim),
    'the': np.random.randn(dim),
    'a': None,  # similar to the
    'mat': np.random.randn(dim),
    'rug': None,  # similar to mat
    'table': np.random.randn(dim),
}

# Make similar words close in embedding space
word_embeddings['cat'] = base_vectors['cat']
word_embeddings['dog'] = base_vectors['cat'] + 0.3 * np.random.randn(dim)
word_embeddings['sat'] = base_vectors['sat']
word_embeddings['sitting'] = base_vectors['sat'] + 0.2 * np.random.randn(dim)
word_embeddings['on'] = base_vectors['on']
word_embeddings['the'] = base_vectors['the']
word_embeddings['a'] = base_vectors['the'] + 0.2 * np.random.randn(dim)
word_embeddings['mat'] = base_vectors['mat']
word_embeddings['rug'] = base_vectors['mat'] + 0.2 * np.random.randn(dim)
word_embeddings['table'] = base_vectors['table']

# Compare: exact match vs paraphrase vs wrong
ref_tokens = ['the', 'cat', 'sat', 'on', 'the', 'mat']
cand_exact = ['the', 'cat', 'sat', 'on', 'the', 'mat']
cand_para = ['a', 'cat', 'sitting', 'on', 'a', 'rug']  # paraphrase
cand_wrong = ['the', 'dog', 'sat', 'on', 'the', 'table']

print(f"Reference: {' '.join(ref_tokens)}\n")

for name, cand_tokens in [('Exact match', cand_exact), ('Paraphrase', cand_para), ('Wrong', cand_wrong)]:
    cand_emb = [word_embeddings[w] for w in cand_tokens]
    ref_emb = [word_embeddings[w] for w in ref_tokens]
    
    # ROUGE-1 for comparison
    r1 = rouge_n(cand_tokens, ref_tokens, 1)
    # BERTScore
    bs = bertscore_simplified(cand_emb, ref_emb)
    
    print(f"{name:>15}: '{' '.join(cand_tokens)}'")
    print(f"                ROUGE-1 F1 = {r1['f1']:.3f}, BERTScore F1 = {bs['f1']:.3f}")

print("\nBERTScore captures that 'rug' is similar to 'mat', 'sitting' to 'sat', etc.")
print("ROUGE only counts exact word overlap.")

## 7. Demo: Evaluate Sample Translations

In [ ]:
# Multiple translation pairs
translation_pairs = [
    {
        'source': 'Der Hund liegt auf dem Boden.',
        'reference': 'the dog is lying on the floor'.split(),
        'system_a': 'the dog lies on the floor'.split(),
        'system_b': 'the dog is on floor'.split(),
        'system_c': 'a hound rests on the ground'.split(),
    },
    {
        'source': 'Das Wetter ist heute sehr schoen.',
        'reference': 'the weather is very nice today'.split(),
        'system_a': 'the weather is very beautiful today'.split(),
        'system_b': 'today weather nice'.split(),
        'system_c': 'it is a beautiful day today'.split(),
    },
    {
        'source': 'Ich habe das Buch gelesen.',
        'reference': 'i have read the book'.split(),
        'system_a': 'i have read the book'.split(),
        'system_b': 'i read book'.split(),
        'system_c': 'the book was read by me'.split(),
    },
]

# Compute corpus-level metrics for each system
systems_bleu = {'System A': [], 'System B': [], 'System C': []}
systems_rouge1 = {'System A': [], 'System B': [], 'System C': []}
systems_rougel = {'System A': [], 'System B': [], 'System C': []}

for pair in translation_pairs:
    ref = [pair['reference']]  # BLEU expects list of references
    for sys_name, sys_key in [('System A', 'system_a'), ('System B', 'system_b'), ('System C', 'system_c')]:
        cand = pair[sys_key]
        b, _, _ = bleu_score(cand, ref, max_n=4)
        r1 = rouge_n(cand, pair['reference'], 1)
        rl = rouge_l(cand, pair['reference'])
        systems_bleu[sys_name].append(b)
        systems_rouge1[sys_name].append(r1['f1'])
        systems_rougel[sys_name].append(rl['f1'])

print(f"{'System':<12} {'Avg BLEU-4':>12} {'Avg ROUGE-1':>14} {'Avg ROUGE-L':>14}")
print("-" * 55)
for sys_name in ['System A', 'System B', 'System C']:
    print(f"{sys_name:<12} {np.mean(systems_bleu[sys_name]):>12.4f} "
          f"{np.mean(systems_rouge1[sys_name]):>14.4f} "
          f"{np.mean(systems_rougel[sys_name]):>14.4f}")

print("\nSystem A: close paraphrases. System B: short/incomplete. System C: restructured.")

In [ ]:
# Visualize metric comparison
fig, ax = plt.subplots(figsize=(9, 5))

metrics = ['BLEU-4', 'ROUGE-1', 'ROUGE-L']
x = np.arange(len(metrics))
width = 0.25
colors = ['#2196F3', '#FF9800', '#4CAF50']

for i, sys_name in enumerate(['System A', 'System B', 'System C']):
    vals = [
        np.mean(systems_bleu[sys_name]),
        np.mean(systems_rouge1[sys_name]),
        np.mean(systems_rougel[sys_name]),
    ]
    ax.bar(x + i * width, vals, width, label=sys_name, color=colors[i], alpha=0.85, edgecolor='white')

ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.set_title('Translation System Comparison')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

## 8. BLEU N-gram Breakdown

In [ ]:
# Show how BLEU changes with n-gram order
ref = ["the quick brown fox jumps over the lazy dog near the river bank".split()]
candidates = {
    'Perfect':  "the quick brown fox jumps over the lazy dog near the river bank".split(),
    'Good':     "the fast brown fox jumped over a lazy dog near the river bank".split(),
    'Mediocre': "a brown fox the over dog lazy jumps river the near bank quick".split(),
    'Bad':      "the the the fox the over the lazy the near the river the".split(),
}

fig, ax = plt.subplots(figsize=(9, 5))
x = [1, 2, 3, 4]
colors_map = {'Perfect': '#2196F3', 'Good': '#4CAF50', 'Mediocre': '#FF9800', 'Bad': '#F44336'}

for name, cand in candidates.items():
    precisions = [modified_precision(cand, ref, n) for n in x]
    ax.plot(x, precisions, 'o-', label=name, color=colors_map[name], linewidth=2, markersize=8)

ax.set_xlabel('N-gram Order')
ax.set_ylabel('Modified Precision')
ax.set_title('N-gram Precision by Order')
ax.set_xticks(x)
ax.set_xticklabels(['Unigram', 'Bigram', 'Trigram', '4-gram'])
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

print("Higher n-gram precisions capture fluency and word order.")
print("The 'Mediocre' candidate has many correct unigrams but bad word order (low bigram+).")

## Interview Questions & Answers

---

**Q: What are the limitations of BLEU?**

1. **No semantic understanding**: "big" and "large" get zero credit. Only exact n-gram matches count.
2. **Precision-only** (with BP hack): does not directly measure recall. The brevity penalty is a rough proxy.
3. **Corpus-level metric**: unreliable at the sentence level (high variance).
4. **No synonym handling**: paraphrases are penalized even if semantically identical.
5. **Equal weight to all n-grams**: content words and function words contribute equally.
6. **Poor correlation with human judgment** for tasks beyond translation (e.g., dialog, creative text).

---

**Q: ROUGE-L vs ROUGE-N?**

- **ROUGE-N** counts exact n-gram matches. ROUGE-2 requires exact bigram overlap.
- **ROUGE-L** uses the Longest Common Subsequence — matches do not need to be contiguous. This captures sentence-level structure even when word order differs.
- ROUGE-L is more robust to reordering: "police killed gunman" vs "gunman killed by police" shares a long subsequence but few bigrams.
- ROUGE-L complexity: O(mn) with DP. ROUGE-N: O(n) with hash maps.

---

**Q: What does perplexity measure?**

Perplexity = exp(cross-entropy) = the effective number of equally likely choices the model faces per token. PPL=100 means the model is as uncertain as if choosing uniformly among 100 options at each step.

- Lower PPL = better model (more confident and correct predictions)
- PPL = vocab_size means random guessing
- PPL = 1 means perfect prediction
- **Caveat**: PPL depends on vocabulary and tokenization. Cannot compare PPL across different tokenizations directly.
- **For LLMs**: typical values are 5-30 for well-trained models on their training domain.

---

**Q: When to use which NLP metric?**

| Task | Primary Metric | Why |
|------|---------------|-----|
| Machine Translation | BLEU, COMET | Precision-oriented, n-gram overlap |
| Summarization | ROUGE | Recall-oriented (did we capture key content?) |
| Language Modeling | Perplexity | Directly measures prediction quality |
| General NLG | BERTScore | Semantic similarity, robust to paraphrasing |

## Quick Reference

```
BLEU = BP * exp(sum w_n * log p_n)
  - Modified precision: clip n-gram counts to max reference count
  - BP = exp(1 - r/c) if c <= r, else 1

ROUGE-N recall = matched n-grams / reference n-grams
ROUGE-L recall = LCS(cand, ref) / len(ref)

Perplexity = exp(cross-entropy) = exp(-1/N * sum log p(w_i))

BERTScore: greedy matching of contextual embeddings -> P/R/F1
```